# <center>Title Topic Modeling</center>
---

In [27]:
import pandas as pd
import nltk
import os
import sys
from nltk.corpus import stopwords
nltk.download('stopwords')

project_root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root_path not in sys.path:
    sys.path.append(project_root_path)
from PreProcessing.pre_processing import PreProcessing
import numpy as np
import spacy as sp

[nltk_data] Downloading package stopwords to /home/ester/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [28]:
df = pd.read_csv("../data/english_youtube_titles")
display(df)

,Unnamed: 0,video_id,title,youtube_video_link,lang
0,1,BFQ8gqvkjz8,nasthug | Boiler Room Tokyo: Tohji Presents u-ha,https://www.youtube.com/watch?v=BFQ8gqvkjz8,en
1,2,X928ajAsdMM,Puddle Of Mudd - Livin' On Borrowed Time (Offi...,https://www.youtube.com/watch?v=X928ajAsdMM,en
2,4,PWbMXDvviTw,The Titans: Rise of the Unstoppable,https://www.youtube.com/watch?v=PWbMXDvviTw,en
3,5,o62RwF_s4Jw,'Rabbi' Shmuley: a walking antisemitic stereotype,https://www.youtube.com/watch?v=o62RwF_s4Jw,en
4,7,QJWplEkDyck,What Just Happened On Our Earth!!! September 2...,https://www.youtube.com/watch?v=QJWplEkDyck,en
...,...,...,...,...,...
257054,429845,cl1QZWF6e_E,Utter Silence Is Fragile,https://www.youtube.com/watch?v=cl1QZWF6e_E,en
257055,429848,cgKtKsefXyQ,Willy Deville - Just Your Friends,https://www.youtube.com/watch?v=cgKtKsefXyQ,en
257056,429849,6qO4TxA7U3c,Putin's Missiles 'Hit' NATO Nation's Train Car...,https://www.youtube.com/watch?v=6qO4TxA7U3c,en
257057,429850,ojLpPQiFguE,"Stop the ""Moonlighting"" Bill. Tharman to Resig...",https://www.youtube.com/watch?v=ojLpPQiFguE,en


In [ ]:
def remove_repetion_caracteres(string, max_repetition=2):
    if not string:
        return string
    
    result = string[0]
    count = 1
    
    for i in range(1, len(string)):
        if string[i] == string[i-1]:
            count += 1
            if count <= max_repetition:
                result += string[i]
        else:
            count = 1
            result += string[i]
    
    return result

def preprocess_text_pipeline(#input_csv_path='./data/dataFrame.csv', 
                              #output_csv_path='./data/dataFrame.csv',
                              df,
                              stopwords_file='stopwords.txt',
                              text_column="comments"):
   
    stem = sp.load("en_core_web_sm")
    pp = PreProcessing(language="en")
    
    custom_stopwords = [line.strip() for line in open(stopwords_file, 'r', encoding='utf-8')]
    english_stopwords = set(stopwords.words('english'))
    
    # Adiciona stopwords à lista da classe PreProcessing
    pp.append_stopwords_list(list(english_stopwords - set(pp.stopwords)) + custom_stopwords)

    def preprocessing(text):
        if pd.isna(text):
            return np.nan

        tokens = stem(text.lower()) # Processo de lematização da biblioteca spaCy - retorna a lista dos tokens do texto
        text = ' '.join([text for token in tokens for text in token.lemma_.strip().split()]) # Junta estes tokens na ordem do texto bruto
        text = pp.remove_stopwords(text) # Remove stopwords presentes
        text = pp.lowercase_unidecode(text) # Coloca tudo em lowercase e remove acento
        text = pp.remove_stopwords(text) # Remove stopwords presentes 
        text = pp.remove_tweet_marking(text) # Remove @ ou # seguido de 1 ou mais carcteres e n ' ' seguidos
        text = remove_repetion_caracteres(text) # Remove a repetição de caracteres ex: gooool -> gool
        text = pp.remove_urls(text) # Remove http\S+ *, ou seja, qualquer http seguido de 1 ou mais caracteres e os espaços no final
        text = pp.remove_punctuation(text) # Remove os sinais de pontuação e reorganiza os espaços
        text = pp.remove_numbers(text) # Remove os números
        text = pp.remove_n(text, n=3) # Remove palavras de tamanho <= n(n=3)
        
        return text

    df['clean_text'] = df[text_column].apply(preprocessing)
    return df

In [30]:
df_clean = preprocess_text_pipeline(df=df, text_column='title')
display(df_clean)

,Unnamed: 0,video_id,title,youtube_video_link,lang,clean_text
0,1,BFQ8gqvkjz8,nasthug | Boiler Room Tokyo: Tohji Presents u-ha,https://www.youtube.com/watch?v=BFQ8gqvkjz8,en,nasthug boiler room tokyo tohji present
1,2,X928ajAsdMM,Puddle Of Mudd - Livin' On Borrowed Time (Offi...,https://www.youtube.com/watch?v=X928ajAsdMM,en,puddle mudd livin borrow time official audio
2,4,PWbMXDvviTw,The Titans: Rise of the Unstoppable,https://www.youtube.com/watch?v=PWbMXDvviTw,en,titans rise unstoppable
3,5,o62RwF_s4Jw,'Rabbi' Shmuley: a walking antisemitic stereotype,https://www.youtube.com/watch?v=o62RwF_s4Jw,en,rabbi shmuley walk antisemitic stereotype
4,7,QJWplEkDyck,What Just Happened On Our Earth!!! September 2...,https://www.youtube.com/watch?v=QJWplEkDyck,en,happen earth september naturaldisaster
...,...,...,...,...,...,...
257054,429845,cl1QZWF6e_E,Utter Silence Is Fragile,https://www.youtube.com/watch?v=cl1QZWF6e_E,en,utter silence fragile
257055,429848,cgKtKsefXyQ,Willy Deville - Just Your Friends,https://www.youtube.com/watch?v=cgKtKsefXyQ,en,willy deville friend
257056,429849,6qO4TxA7U3c,Putin's Missiles 'Hit' NATO Nation's Train Car...,https://www.youtube.com/watch?v=6qO4TxA7U3c,en,putin missile nato nation train carry atacms s...
257057,429850,ojLpPQiFguE,"Stop the ""Moonlighting"" Bill. Tharman to Resig...",https://www.youtube.com/watch?v=ojLpPQiFguE,en,stop moonlighting bill tharman resign online z...


In [ ]:
#df_clean.to_csv("../data/preprocessed_english_titles")